In [ ]:
import cv2
import numpy as np
import time

try:
    mtx = np.array(np.loadtxt('calibration/calib_mtx.csv', delimiter=','))
    dist = np.array(np.loadtxt('calibration/calib_dist.csv', delimiter=','))
except FileNotFoundError:
    print('Files not found, make sure you ran calibration!')

aruco_dict = cv2.aruco.getPredefinedDictionary(
    cv2.aruco.DICT_4X4_50
)

# detector = cv2.aruco.ArucoDetector(aruco_dict)

tagSize = 38 / 1000 # m

def drawSquare(frame, corners):
    for i in range(4):
        cv2.line(frame, (int(corners[0][i][0]),int(corners[0][i][1])), (int(corners[0][(i+1)%4][0]), int(corners[0][(i+1)%4][1])), (0,0,255), 3)

In [ ]:
cap = cv2.VideoCapture(0)
while True:
    ret, frame = cap.read()
    if (not ret): continue

    # corners, ids, _ = detector.detectMarkers(frame)
    corners, ids, _ = cv2.aruco.detectMarkers(frame, aruco_dict)

    objpoints = np.array([
        (-tagSize/2, tagSize/2, 0),
        (tagSize/2, tagSize/2, 0),
        (tagSize/2, -tagSize/2, 0),
        (-tagSize/2, -tagSize/2, 0)
    ])

    # rvecs = []
    # tvecs = []
    
    if corners is not None and ids is not None:
        for square, id in zip(corners, ids):
            drawSquare(frame, square)
            ret, rvec, tvec = cv2.solvePnP(objpoints, square[0], mtx, dist)
            # rvecs.append(rvec)
            # tvecs.append(tvec)
            distance = np.linalg.norm(tvec) 
            cv2.putText(frame, '%f mm' % (distance * 1000), (int(square[0][0][0]), int(square[0][0][1])), cv2.FONT_HERSHEY_COMPLEX, 1, (60,255,70), 2)
            cv2.drawFrameAxes(frame, mtx, dist, rvec, tvec, tagSize * 1.5, 2)

            # TODO: calculate location of centroid based on dimensions of block and pose vector
            # Centroid is (block length / 2) along block's pose vectors rx, ry, rz

    cv2.imshow('ArUco', frame)

    key = cv2.waitKey(1)

    if key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
cv2.waitKey(1)

-1